# Hard-wall purification alpha sweep — lane A

Endpoint-only occupation-derived finite-time single-particle Lyapunov diagnostics. This is **not** a tangent-cocycle or Choi spectrum. The two notebooks own disjoint complete `(Ny, alpha_1)` configurations and may run concurrently. Results are immutable five-trajectory shards; state and RNG are checkpointed every ten cycles.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import torch

LANE = 'A'
REPORT_ONLY = False
# Qualification default: run exactly one worst-case execution batch.
# After it succeeds below 38 GiB, set this to None for the full lane.
MAX_NEW_EXECUTION_BATCHES = 1
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/18_hard_wall_purification_alpha_endpoint')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs/hard_wall_maxmix_purification_alpha21_ny24-50_s100_2ny_endpoint_lyapunov_v1')
LOCAL_BUNDLE_DIR = Path(f'/content/18_hard_wall_purification_alpha_endpoint_lane_{LANE.lower()}')
SCRATCH_ROOT = Path(f'/content/hard_wall_purification_alpha_endpoint_scratch_lane_{LANE.lower()}')
CONFIG = {'Nx': 20, 'Ny_values': [24, 28, 32, 36, 40, 44, 50], 'alpha_1_values': [1.0, 1.2, 1.4, 1.6, 1.7, 1.8, 1.85, 1.9, 1.95, 1.975, 2.0, 2.025, 2.05, 2.1, 2.15, 2.2, 2.3, 2.4, 2.6, 2.8, 3.0], 'alpha_2': 30.0, 'backend': 'local', 'construction': {'DW': True, 'dw_truncation': True, 'meas_slab_only': True, 'name': 'hard_support_truncated', 'wall_locations': [5, 15]}, 'cycles_multiplier': 2, 'device': 'cuda:0', 'dtype': 'complex128', 'execution_batch_size_by_Ny': {'24': 100, '28': 100, '32': 100, '36': 90, '40': 75, '44': 60, '50': 50}, 'filling_fraction': 0.5, 'gpu_memory_hard_limit_gib': 38.0, 'init_mode': 'maxmix', 'n_a': 0.5, 'nshell': 1, 'observables': {'Ny50_full_centered_spectrum': True, 'Ny50_selected_eigenvectors': 'abs(a)<=0.9', 'cap_tolerance': 1e-09, 'centered_half_gap': True, 'eigenvector_endpoint_margin': 0.1, 'lyapunov_formula': 'lambda=(log(1-a)-log(1+a))/(2*T)', 'lyapunov_half_gap': True, 'lyapunov_two_sided_gap': True, 'nearest_mode_count': 16, 'save_covariance_history': False, 'save_final_covariance': False, 'spectrum_cycle': '2Ny endpoint only'}, 'observer_sample_chunk_by_Ny': {'24': 20, '28': 20, '32': 10, '36': 10, '40': 5, '44': 5, '50': 2}, 'perfect_correction': True, 'postselect': False, 'result_shard_size': 5, 'resume_initial_purity_tolerance': 0.50000001, 'root_seed': 2026091701, 'sampling_revision': 'hard_wall_maxmix_purification_alpha21_ny24-50_s100_2ny_endpoint_lyapunov_v1', 'samples_per_case': 100, 'segment_cycles': 10, 'sequence': 'raster_y', 'state_representation': 'covariance', 'trial_orbitals': 'X'}

if not torch.cuda.is_available():
    raise RuntimeError('This production notebook requires CUDA.')
gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if 'A100' not in gpu_name or gpu_gib < 38.0:
    raise RuntimeError(f'Expected an A100 40-GB-class GPU, got {gpu_name} ({gpu_gib:.2f} GiB)')
if torch.complex128 != getattr(torch, CONFIG['dtype']):
    raise RuntimeError('Locked dtype is not complex128.')
print(json.dumps({
    'lane': LANE, 'config': CONFIG,
    'bundle': str(BUNDLE_DRIVE_DIR), 'output': str(OUTPUT_ROOT),
    'scratch': str(SCRATCH_ROOT), 'gpu': gpu_name, 'gpu_GiB': gpu_gib,
    'gpu_memory_hard_limit_GiB': CONFIG['gpu_memory_hard_limit_gib'],
    'report_only': REPORT_ONLY,
    'max_new_execution_batches': MAX_NEW_EXECUTION_BATCHES,
}, indent=2, sort_keys=True), flush=True)


In [ ]:
import importlib.util
import shutil
import sys

if not BUNDLE_DRIVE_DIR.is_dir():
    raise FileNotFoundError(f'Missing Drive bundle: {BUNDLE_DRIVE_DIR}')
if LOCAL_BUNDLE_DIR.exists():
    shutil.rmtree(LOCAL_BUNDLE_DIR)
shutil.copytree(BUNDLE_DRIVE_DIR, LOCAL_BUNDLE_DIR)
config_path = Path(f'/content/hard_wall_purification_alpha_endpoint_lane_{LANE.lower()}.json')
config_path.write_text(json.dumps(CONFIG, indent=2, sort_keys=True) + '\n', encoding='utf-8')
runner_args = [
    '--config', str(config_path), '--lane', LANE,
    '--output-root', str(OUTPUT_ROOT), '--scratch-root', str(SCRATCH_ROOT),
]
if REPORT_ONLY:
    runner_args.append('--report-only')
if MAX_NEW_EXECUTION_BATCHES is not None:
    runner_args.extend(['--max-new-execution-batches', str(int(MAX_NEW_EXECUTION_BATCHES))])
runner_path = LOCAL_BUNDLE_DIR / 'run_campaign.py'
print('[launch in notebook kernel] ' + str(runner_path) + ' ' + ' '.join(runner_args), flush=True)
module_name = f'_hard_wall_purification_alpha_endpoint_lane_{LANE.lower()}'
sys.modules.pop(module_name, None)
sys.path.insert(0, str(LOCAL_BUNDLE_DIR))
spec = importlib.util.spec_from_file_location(module_name, runner_path)
if spec is None or spec.loader is None:
    raise RuntimeError(f'Could not load staged runner: {runner_path}')
runner = importlib.util.module_from_spec(spec)
sys.modules[module_name] = runner
spec.loader.exec_module(runner)
returncode = runner.main(runner_args)
if returncode:
    raise RuntimeError(f'Runner exited with status {returncode}')
print(f'[notebook] lane {LANE} runner exited successfully', flush=True)


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
